# RAG Pipeline — Chatbot Hukum Indonesia dengan LLM Fine-Tuned

**Modul**: Pengembangan Generative AI berbasis LLM — Digdaya Hackathon BI

**Target**: Advanced (4 pts)

---

## Ringkasan Notebook

Pipeline RAG lengkap dengan seluruh fitur Advanced:

1. **PDF Loading** — 4 dokumen UU Indonesia
2. **Metadata Enrichment** — Ekstraksi nomor UU, tahun, bab, pasal
3. **Text Splitting** — Parent-Child chunks (512/1024 tokens)
4. **Ensemble Retriever** — BM25 (weight 0.3) + Semantic ChromaDB (weight 0.7)
5. **Parent-Child Retriever** — Child search → Parent context
6. **HyDE** — 2 hallucinated answers untuk query transformation
7. **Reranker** — `BAAI/bge-reranker-v2-m3` → Top-3 chunks
8. **DuckDuckGo Fallback** — Jika Reranker score < threshold (0.5)
9. **Metadata Filtering + Citation** — Sumber UU, pasal, halaman
10. **Gradio ChatInterface** — UI interaktif

> **Catatan**: Notebook ini didesain untuk Google Colab dengan GPU T4. Download PDF dari Google Drive sebelum menjalankan.
> Link PDF: https://drive.google.com/drive/folders/... (dari ketentuan submission)

## 1. Instalasi Dependencies

In [ ]:
# Install semua dependencies
%%capture
!pip install unsloth
!pip install transformers datasets accelerate peft bitsandbytes
!pip install sentence-transformers chromadb
!pip install langchain langchain-community
!pip install pypdf pypdf2
!pip install rank-bm25
!pip install FlagEmbedding
!pip install gradio
!pip install duckduckgo-search
!pip install rouge-score nltk
!pip install huggingface_hub safetensors

In [ ]:
import unsloth  # noqa: F401 — init Unsloth optimizations (harus paling awal)

import os
import re
import gc
from typing import List, Dict, Tuple, Optional
from pathlib import Path

import torch
import numpy as np
from tqdm import tqdm

# PDF
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

# Embedding & Vector DB
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings as ChromaSettings

# BM25
from rank_bm25 import BM25Okapi

# Reranker
from FlagEmbedding import FlagReranker

# DuckDuckGo fallback
from duckduckgo_search import DDGS

# LLM
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# NLP utilities
import nltk
try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab", quiet=True)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: GPU tidak terdeteksi! Pastikan Runtime → Change runtime type → T4 GPU")

## 2. Load 4 Dokumen PDF

Download PDF dari Google Drive terlebih dahulu, lalu letakkan di folder `./pdfs/`.

In [ ]:
# Konfigurasi path PDF (relative dari lokasi notebook di folder PGABL_Rayhan/)
PDF_DIR = "./Document RAG/Document RAG"  # Folder tempat menyimpan 4 dokumen UU

# List semua PDF
pdf_files = sorted([f for f in os.listdir(PDF_DIR) if f.endswith(".pdf")])
print(f"Ditemukan {len(pdf_files)} file PDF:")
for i, f in enumerate(pdf_files):
    filepath = os.path.join(PDF_DIR, f)
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"  {i+1}. {f} ({size_mb:.1f} MB)")

if len(pdf_files) == 0:
    print()
    print("=" * 60)
    print("PERHATIAN: Belum ada file PDF di folder ./Document RAG/Document RAG/")
    print("Download 4 dokumen UU dari Google Drive terlebih dahulu.")
    print("=" * 60)

In [ ]:
# Load semua PDF menggunakan PyPDFLoader
raw_documents = []

for pdf_file in tqdm(pdf_files, desc="Loading PDFs"):
    filepath = os.path.join(PDF_DIR, pdf_file)
    loader = PyPDFLoader(filepath)
    pages = loader.load()
    
    # Tambahkan metadata: nama file
    for page in pages:
        page.metadata["source_file"] = pdf_file
        page.metadata["page_number"] = page.metadata.get("page", 0)
    
    raw_documents.extend(pages)
    print(f"  {pdf_file}: {len(pages)} halaman")

print(f"\nTotal dokumen: {len(raw_documents)} halaman")

## 3. Metadata Enrichment

Ekstrak informasi dari teks UU: nomor UU, tahun, judul, bab, dan pasal. Metadata ini digunakan untuk filtering dan sitasi.

In [ ]:
def extract_document_metadata(text: str, source_file: str) -> Dict[str, any]:
    """
    Ekstrak metadata dari teks UU Indonesia.
    
    Mencari pola:
    - "UNDANG-UNDANG REPUBLIK INDONESIA NOMOR X TAHUN Y"
    - "PERATURAN PEMERINTAH REPUBLIK INDONESIA NOMOR X TAHUN Y"
    - "BAB I/II/..." atau "BAB 1/2/..."
    - "Pasal X"
    """
    metadata = {
        "document_type": "",  # UU, PP, dll
        "document_number": "",
        "document_year": "",
        "document_title": "",
        "chapter": "",
        "article": "",
        "source_file": source_file,
    }
    
    # Cari tipe dan nomor dokumen
    # Pola: "UNDANG-UNDANG ... NOMOR X TAHUN Y"
    uu_pattern = r"(UNDANG-UNDANG|PERATURAN PEMERINTAH|PERATURAN PRESIDEN|KEPUTUSAN PRESIDEN)\s+(?:REPUBLIK INDONESIA\s+)?(?:NOMOR|NO[.:]?)\s+(\d+)\s+(?:TAHUN)\s+(\d{4})"
    match = re.search(uu_pattern, text, re.IGNORECASE)
    if match:
        metadata["document_type"] = match.group(1).upper()
        metadata["document_number"] = match.group(2)
        metadata["document_year"] = match.group(3)
    
    # Cari judul UU (biasanya setelah "TENTANG")
    tentang_pattern = r"TENTANG\s+(.+?)(?:\n|DENGAN RAHMAT|PRESIDEN REPUBLIK INDONESIA)"
    match = re.search(tentang_pattern, text, re.IGNORECASE)
    if match:
        metadata["document_title"] = match.group(1).strip()
    
    # Cari BAB
    bab_pattern = r"BAB\s+([IVXLCDM]+|[0-9]+)\s*(?::|\.|–|-|\s)\s*(.+?)(?:\n|Pasal|Bagian)"
    match = re.search(bab_pattern, text, re.IGNORECASE)
    if match:
        metadata["chapter"] = f"BAB {match.group(1)} — {match.group(2).strip()}"
    
    # Cari Pasal
    pasal_pattern = r"Pasal\s+(\d+[A-Za-z]?)"
    matches = re.findall(pasal_pattern, text, re.IGNORECASE)
    if matches:
        # Ambil pasal pertama yang ditemukan di halaman ini
        metadata["article"] = f"Pasal {matches[0]}"
        if len(matches) > 1:
            metadata["articles_mentioned"] = ", ".join([f"Pasal {m}" for m in matches[:5]])
    
    return metadata

print("Fungsi extract_document_metadata() siap.")

In [ ]:
# Enrich metadata untuk setiap halaman
for doc in tqdm(raw_documents, desc="Enriching metadata"):
    enriched = extract_document_metadata(doc.page_content, doc.metadata.get("source_file", ""))
    doc.metadata.update(enriched)

# Tampilkan contoh metadata
print("\nContoh metadata hasil enrichment:")
print("=" * 60)
for i in [0, len(raw_documents)//3, len(raw_documents)//2]:
    doc = raw_documents[i]
    print(f"Halaman {i+1}:")
    print(f"  File: {doc.metadata.get('source_file', 'N/A')}")
    print(f"  Tipe: {doc.metadata.get('document_type', 'N/A')}")
    print(f"  Nomor: {doc.metadata.get('document_number', 'N/A')}")
    print(f"  Tahun: {doc.metadata.get('document_year', 'N/A')}")
    print(f"  Judul: {doc.metadata.get('document_title', 'N/A')[:80]}..." if doc.metadata.get('document_title') else "")
    print(f"  Bab: {doc.metadata.get('chapter', 'N/A')}")
    print(f"  Pasal: {doc.metadata.get('article', 'N/A')}")
    print()

## 4. Text Splitting — Parent-Child Strategy

- **Child chunks**: 512 tokens, overlap 128 — untuk vector search (embedding)
- **Parent chunks**: 1024 tokens — untuk LLM context (diberikan ke model)

In [ ]:
# Text splitter untuk child chunks (pencarian vektor)
CHILD_CHUNK_SIZE = 512
CHILD_CHUNK_OVERLAP = 128

# Text splitter untuk parent chunks (konteks LLM)
PARENT_CHUNK_SIZE = 1024
PARENT_CHUNK_OVERLAP = 256

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHILD_CHUNK_SIZE,
    chunk_overlap=CHILD_CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=PARENT_CHUNK_SIZE,
    chunk_overlap=PARENT_CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

# Split menjadi child dan parent chunks
child_docs = child_splitter.split_documents(raw_documents)
parent_docs = parent_splitter.split_documents(raw_documents)

print(f"Child chunks: {len(child_docs)} (size={CHILD_CHUNK_SIZE}, overlap={CHILD_CHUNK_OVERLAP})")
print(f"Parent chunks: {len(parent_docs)} (size={PARENT_CHUNK_SIZE}, overlap={PARENT_CHUNK_OVERLAP})")

# Assign unique ID untuk setiap chunk (untuk parent-child mapping)
for i, doc in enumerate(child_docs):
    doc.metadata["chunk_id"] = f"child_{i}"
    doc.metadata["chunk_type"] = "child"

for i, doc in enumerate(parent_docs):
    doc.metadata["chunk_id"] = f"parent_{i}" 
    doc.metadata["chunk_type"] = "parent"

In [ ]:
# Buat mapping parent-child berdasarkan overlap konten
# Child → Parent: cari parent yang kontennya mengandung child
def build_parent_child_map(child_docs: List[Document], parent_docs: List[Document]) -> Dict[str, str]:
    """
    Membuat mapping dari child chunk ID ke parent chunk ID.
    
    Strategi: untuk setiap child chunk, cari parent chunk yang 
    memiliki halaman dan file sumber yang sama.
    """
    child_to_parent = {}
    
    # Build index parent by (source_file, page_number)
    parent_index = {}
    for p_doc in parent_docs:
        key = (p_doc.metadata.get("source_file", ""), p_doc.metadata.get("page_number", 0))
        if key not in parent_index:
            parent_index[key] = []
        parent_index[key].append(p_doc)
    
    for c_doc in child_docs:
        c_key = (c_doc.metadata.get("source_file", ""), c_doc.metadata.get("page_number", 0))
        # Cari parent dengan halaman yang sama
        if c_key in parent_index:
            # Ambil parent terpanjang (paling komprehensif)
            best_parent = max(parent_index[c_key], key=lambda d: len(d.page_content))
            child_to_parent[c_doc.metadata["chunk_id"]] = best_parent.metadata["chunk_id"]
        else:
            # Fallback: gunakan parent pertama
            child_to_parent[c_doc.metadata["chunk_id"]] = parent_docs[0].metadata["chunk_id"]
    
    return child_to_parent

child_to_parent_map = build_parent_child_map(child_docs, parent_docs)
print(f"Parent-Child mapping: {len(child_to_parent_map)} child → parent links")

## 5. Embedding & Vector Store (ChromaDB)

Menggunakan `BAAI/bge-m3` — open-source multilingual embedding model.

In [ ]:
# Load embedding model open-source
EMBEDDING_MODEL_NAME = "BAAI/bge-m3"

print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")
print(f"Max sequence length: {embedding_model.max_seq_length}")

In [ ]:
# Buat embeddings untuk semua child chunks
print("Membuat embeddings untuk child chunks...")

child_texts = [doc.page_content for doc in child_docs]

# Encode dalam batch untuk efisiensi
BATCH_SIZE = 32
embeddings = []

for i in tqdm(range(0, len(child_texts), BATCH_SIZE), desc="Encoding"):
    batch = child_texts[i:i+BATCH_SIZE]
    batch_embeddings = embedding_model.encode(
        batch,
        normalize_embeddings=True,  # Cosine similarity
        show_progress_bar=False,
    )
    embeddings.extend(batch_embeddings)

embeddings = np.array(embeddings)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
# Simpan ke ChromaDB
print("Menyimpan ke ChromaDB...")

chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Hapus collection jika sudah ada (fresh start)
try:
    chroma_client.delete_collection("indonesian_laws")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="indonesian_laws",
    metadata={"hnsw:space": "cosine"},
)

# Masukkan data ke ChromaDB dalam batch
CHROMA_BATCH = 100
for i in tqdm(range(0, len(child_docs), CHROMA_BATCH), desc="Inserting to ChromaDB"):
    end_idx = min(i + CHROMA_BATCH, len(child_docs))
    
    batch_docs = child_docs[i:end_idx]
    batch_ids = [doc.metadata["chunk_id"] for doc in batch_docs]
    batch_embeddings = embeddings[i:end_idx].tolist()
    batch_texts = [doc.page_content for doc in batch_docs]
    batch_metadatas = [
        {k: str(v) for k, v in doc.metadata.items()}
        for doc in batch_docs
    ]
    
    collection.add(
        ids=batch_ids,
        embeddings=batch_embeddings,
        documents=batch_texts,
        metadatas=batch_metadatas,
    )

print(f"ChromaDB collection: {collection.count()} documents")

## 6. BM25 Index

BM25 untuk pencarian berbasis keyword (lexical search). Digunakan bersama semantic search dalam Ensemble Retriever.

In [ ]:
# Tokenisasi untuk BM25 (Bahasa Indonesia sederhana: split by whitespace)
def tokenize_id(text: str) -> List[str]:
    """Tokenisasi sederhana untuk Bahasa Indonesia."""
    # Lowercase + split by non-alphanumeric
    text = text.lower()
    tokens = re.findall(r'[a-z0-9]+', text)
    return tokens

# Build BM25 corpus dari child chunks
bm25_corpus = [tokenize_id(doc.page_content) for doc in child_docs]
bm25 = BM25Okapi(bm25_corpus)

print(f"BM25 index built: {len(bm25_corpus)} documents")

## 7. Load Model Fine-Tuned / GRPO

Load model hasil fine-tuning (atau GRPO jika sudah) untuk inference RAG.

In [ ]:
HF_USERNAME = "RayhanLup1n"

# Pilih model: gunakan model GRPO jika sudah dilatih, jika belum gunakan model fine-tuned
# MODEL_PATH = f"{HF_USERNAME}/qwen2.5-7b-indonesian-legal-grpo"  # GRPO model
MODEL_PATH = f"{HF_USERNAME}/qwen2.5-7b-indonesian-legal-finetuned"  # Fine-tuned model

# Atau gunakan model lokal:
# MODEL_PATH = "./qwen2.5-7b-indonesian-legal-finetuned"

print(f"Loading model: {MODEL_PATH}")

llm_model, llm_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,  # 4-bit untuk hemat VRAM saat inference
)

# Cek apakah model support GRPO thinking format
FastLanguageModel.for_inference(llm_model)

print("Model LLM siap untuk inference.")

## 8. Ensemble Retriever — BM25 + Semantic

Menggabungkan pencarian BM25 (lexical) dengan semantic search (ChromaDB).
- BM25 weight: 0.3
- Semantic weight: 0.7
- Top-K: 5 dokumen
- Menggunakan Reciprocal Rank Fusion (RRF)

In [ ]:
def ensemble_retrieve(
    query: str,
    top_k: int = 5,
    bm25_weight: float = 0.3,
    semantic_weight: float = 0.7,
    metadata_filter: Optional[Dict[str, str]] = None,
) -> List[Document]:
    """
    Ensemble Retriever: menggabungkan BM25 (lexical) + Semantic (ChromaDB).
    
    Menggunakan Reciprocal Rank Fusion (RRF) untuk menggabungkan hasil.
    
    Args:
        query: Query pencarian
        top_k: Jumlah dokumen yang diambil
        bm25_weight: Bobot untuk BM25 score
        semantic_weight: Bobot untuk semantic score
        metadata_filter: Filter metadata (misal: {"document_number": "13"})
    
    Returns:
        List[Document]: Dokumen hasil retrieve (deduplicated)
    """
    # 1. BM25 Search
    bm25_tokens = tokenize_id(query)
    bm25_scores = bm25.get_scores(bm25_tokens)
    
    # Normalisasi BM25 scores ke [0, 1]
    bm25_max = max(bm25_scores) if max(bm25_scores) > 0 else 1
    bm25_scores_norm = bm25_scores / bm25_max
    
    # 2. Semantic Search (ChromaDB)
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
    )[0].tolist()
    
    # Query ChromaDB (dengan metadata filter jika ada)
    where_clause = None
    if metadata_filter:
        where_clause = {k: v for k, v in metadata_filter.items()}
    
    chroma_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k * 3,  # Ambil lebih banyak untuk RRF
        where=where_clause,
        include=["documents", "metadatas", "distances"],
    )
    
    # 3. Reciprocal Rank Fusion (RRF)
    # Hitung RRF score: sum(weight / (k + rank)) for each retriever
    K = 60  # RRF constant
    
    # Buat mapping chunk_id -> document untuk lookup cepat
    child_doc_map = {doc.metadata["chunk_id"]: doc for doc in child_docs}
    
    # BM25 ranking
    bm25_ranking = sorted(
        enumerate(bm25_scores_norm),
        key=lambda x: x[1],
        reverse=True
    )
    bm25_ranks = {}
    for rank, (idx, score) in enumerate(bm25_ranking[:top_k * 3]):
        chunk_id = child_docs[idx].metadata["chunk_id"]
        bm25_ranks[chunk_id] = rank + 1
    
    # Semantic ranking
    semantic_ranks = {}
    for rank, (chunk_id, distance) in enumerate(zip(
        chroma_results["ids"][0],
        chroma_results["distances"][0]
    )):
        semantic_ranks[chunk_id] = rank + 1
    
    # Hitung RRF score
    all_chunk_ids = set(list(bm25_ranks.keys()) + list(semantic_ranks.keys()))
    rrf_scores = {}
    for chunk_id in all_chunk_ids:
        bm25_rrf = bm25_weight / (K + bm25_ranks.get(chunk_id, len(child_docs)))
        semantic_rrf = semantic_weight / (K + semantic_ranks.get(chunk_id, len(child_docs)))
        rrf_scores[chunk_id] = bm25_rrf + semantic_rrf
    
    # Sort by RRF score dan ambil top_k
    sorted_chunks = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    
    # Ambil dokumen
    results = []
    for chunk_id, rrf_score in sorted_chunks:
        doc = child_doc_map.get(chunk_id)
        if doc:
            doc.metadata["rrf_score"] = rrf_score
            results.append(doc)
    
    return results

print("Fungsi ensemble_retrieve() siap.")
print(f"  BM25 weight: 0.3 | Semantic weight: 0.7 | Top-K: 5")

## 9. Parent-Child Retriever

Setelah child chunks di-retrieve via Ensemble Retriever, ambil parent chunks yang sesuai untuk memberikan konteks lebih luas ke LLM.

In [ ]:
def parent_child_retrieve(
    child_results: List[Document],
) -> List[Document]:
    """
    Parent-Child Retriever: dari child chunk hasil retrieve,
    ambil parent chunk yang sesuai untuk konteks LLM.
    
    Args:
        child_results: Hasil retrieve dari Ensemble Retriever (child chunks)
    
    Returns:
        List[Document]: Parent documents (deduplicated)
    """
    # Build parent lookup map
    parent_map = {doc.metadata["chunk_id"]: doc for doc in parent_docs}
    
    # Ambil parent untuk setiap child
    parent_ids_seen = set()
    parent_results = []
    
    for child_doc in child_results:
        child_id = child_doc.metadata.get("chunk_id")
        parent_id = child_to_parent_map.get(child_id)
        
        if parent_id and parent_id not in parent_ids_seen:
            parent_ids_seen.add(parent_id)
            parent_doc = parent_map.get(parent_id)
            if parent_doc:
                # Salin metadata dari child yang relevan
                parent_doc.metadata["retrieved_via_child"] = child_id
                parent_doc.metadata["child_rrf_score"] = child_doc.metadata.get("rrf_score", 0)
                parent_results.append(parent_doc)
    
    return parent_results

print("Fungsi parent_child_retrieve() siap.")

## 10. HyDE — Hypothetical Document Embeddings

Generate 2 jawaban halusinasi dari LLM untuk query, lalu gunakan embedding dari jawaban tersebut untuk memperkaya pencarian.

In [ ]:
def generate_hyde_answers(query: str, num_answers: int = 2) -> List[str]:
    """
    Generate hypothetical (halusinasi) answers untuk query.
    
    LLM diminta untuk "berpura-pura" menjawab pertanyaan hukum
    tanpa diberikan dokumen referensi. Jawaban ini digunakan
    untuk memperkaya pencarian vektor.
    
    Args:
        query: Pertanyaan user
        num_answers: Jumlah jawaban halusinasi (minimal 2)
    
    Returns:
        List[str]: Jawaban halusinasi
    """
    hyde_prompt = (
        f"Buatlah jawaban hipotetis (berdasarkan pengetahuan umum saja, "
        f"tanpa referensi spesifik) untuk pertanyaan hukum berikut:\n\n"
        f"{query}\n\n"
        f"Jawaban hipotetis:"
    )
    
    answers = []
    for i in range(num_answers):
        # Tokenize dengan ChatML
        messages = [
            {"from": "user", "value": hyde_prompt},
        ]
        inputs = llm_tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        ).to("cuda")
        
        outputs = llm_model.generate(
            input_ids=inputs,
            max_new_tokens=200,
            temperature=0.9,  # Variasi lebih tinggi untuk jawaban berbeda
            do_sample=True,
            top_p=0.95,
            pad_token_id=llm_tokenizer.pad_token_id,
        )
        
        answer = llm_tokenizer.decode(
            outputs[0][inputs.shape[1]:],
            skip_special_tokens=True,
        )
        answers.append(answer.strip())
    
    return answers

print("Fungsi generate_hyde_answers() siap.")

In [ ]:
def hyde_retrieve(query: str, top_k: int = 5) -> List[Document]:
    """
    HyDE Retrieve: gabungkan query asli + 2 jawaban halusinasi,
    rata-rata embedding mereka, lalu retrieve dengan ensemble retriever.
    """
    # Generate 2 jawaban halusinasi
    hyde_answers = generate_hyde_answers(query, num_answers=2)
    
    # Gabungkan query + jawaban untuk embedding
    hyde_texts = [query] + hyde_answers
    
    # Embed semua teks
    hyde_embeddings = embedding_model.encode(
        hyde_texts,
        normalize_embeddings=True,
    )
    
    # Rata-rata embedding (HyDE query vector)
    hyde_query_vector = np.mean(hyde_embeddings, axis=0)
    hyde_query_vector = hyde_query_vector / np.linalg.norm(hyde_query_vector)
    
    # Gunakan averaged embedding untuk semantic search + BM25 dengan query asli
    # Semantic: HyDE vector
    chroma_results = collection.query(
        query_embeddings=[hyde_query_vector.tolist()],
        n_results=top_k * 3,
        include=["documents", "metadatas", "distances"],
    )
    
    # BM25: query asli
    bm25_tokens = tokenize_id(query)
    bm25_scores = bm25.get_scores(bm25_tokens)
    bm25_max = max(bm25_scores) if max(bm25_scores) > 0 else 1
    bm25_scores_norm = bm25_scores / bm25_max
    
    # RRF combining HyDE semantic + BM25
    K = 60
    child_doc_map = {doc.metadata["chunk_id"]: doc for doc in child_docs}
    
    # BM25 ranks
    bm25_ranking = sorted(enumerate(bm25_scores_norm), key=lambda x: x[1], reverse=True)
    bm25_ranks = {}
    for rank, (idx, score) in enumerate(bm25_ranking[:top_k * 3]):
        chunk_id = child_docs[idx].metadata["chunk_id"]
        bm25_ranks[chunk_id] = rank + 1
    
    # Semantic ranks
    semantic_ranks = {}
    for rank, chunk_id in enumerate(chroma_results["ids"][0]):
        semantic_ranks[chunk_id] = rank + 1
    
    # RRF
    all_ids = set(list(bm25_ranks.keys()) + list(semantic_ranks.keys()))
    rrf_scores = {}
    for chunk_id in all_ids:
        bm25_rrf = 0.3 / (K + bm25_ranks.get(chunk_id, len(child_docs)))
        semantic_rrf = 0.7 / (K + semantic_ranks.get(chunk_id, len(child_docs)))
        rrf_scores[chunk_id] = bm25_rrf + semantic_rrf
    
    sorted_chunks = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    
    results = []
    for chunk_id, rrf_score in sorted_chunks:
        doc = child_doc_map.get(chunk_id)
        if doc:
            doc.metadata["rrf_score"] = rrf_score
            results.append(doc)
    
    return results, hyde_answers

print("Fungsi hyde_retrieve() siap.")

## 11. Reranker

Menggunakan `BAAI/bge-reranker-v2-m3` — Cross-Encoder multilingual untuk mengurutkan ulang chunk berdasarkan relevansi dengan query.

In [ ]:
# Load Reranker model
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"

print(f"Loading reranker: {RERANKER_MODEL_NAME}...")
reranker = FlagReranker(RERANKER_MODEL_NAME, use_fp16=torch.cuda.is_available())
print("Reranker loaded.")

In [ ]:
def rerank_documents(
    query: str,
    documents: List[Document],
    top_k: int = 3,
) -> Tuple[List[Document], List[float]]:
    """
    Rerank dokumen menggunakan Cross-Encoder reranker.
    
    Args:
        query: Query user
        documents: Dokumen yang akan di-rerank
        top_k: Jumlah dokumen teratas yang dikembalikan
    
    Returns:
        Tuple[List[Document], List[float]]: (reranked documents, relevance scores)
    """
    if not documents:
        return [], []
    
    # Buat pasangan [query, document] untuk reranker
    pairs = [[query, doc.page_content[:1000]] for doc in documents]  # Batasi 1000 chars
    
    # Hitung relevance scores
    scores = reranker.compute_score(pairs, normalize=True)
    
    # Pastikan scores dalam format list
    if isinstance(scores, float):
        scores = [scores]
    
    # Sort document by score descending
    scored_docs = list(zip(documents, scores))
    scored_docs.sort(key=lambda x: x[1], reverse=True)
    
    # Ambil top_k
    top_docs = [doc for doc, score in scored_docs[:top_k]]
    top_scores = [score for doc, score in scored_docs[:top_k]]
    
    # Simpan relevance score ke metadata
    for doc, score in zip(top_docs, top_scores):
        doc.metadata["reranker_score"] = score
    
    return top_docs, top_scores

print("Fungsi rerank_documents() siap.")

## 12. DuckDuckGo Fallback

Jika relevance score dari Top-1 Reranker di bawah threshold (0.5), system beralih ke DuckDuckGo Search untuk mencari informasi dari internet.

In [ ]:
RELEVANCE_THRESHOLD = 0.5  # Threshold untuk fallback

def duckduckgo_search(query: str, max_results: int = 3) -> List[Dict[str, str]]:
    """
    Cari informasi dari internet menggunakan DuckDuckGo.
    
    Args:
        query: Query pencarian
        max_results: Maksimal hasil pencarian
    
    Returns:
        List[Dict]: List hasil pencarian {title, body, href}
    """
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(f"{query} Indonesia hukum", max_results=max_results))
            return results
    except Exception as e:
        print(f"[WARNING] DuckDuckGo search failed: {e}")
        return []

print(f"DuckDuckGo fallback siap. Threshold: {RELEVANCE_THRESHOLD}")

## 13. Format Citation

Menghasilkan sitasi dari metadata dokumen untuk ditampilkan bersama jawaban.

In [ ]:
def format_citation(doc: Document) -> str:
    """
    Format sitasi dari metadata dokumen.
    
    Format: "UU/PP No. X Tahun Y, [BAB Z], Pasal A, Hlm. B (File: C)"
    """
    parts = []
    
    doc_type = doc.metadata.get("document_type", "")
    doc_num = doc.metadata.get("document_number", "")
    doc_year = doc.metadata.get("document_year", "")
    
    if doc_type and doc_num and doc_year:
        parts.append(f"{doc_type} No. {doc_num} Tahun {doc_year}")
    elif doc_type:
        parts.append(doc_type)
    
    chapter = doc.metadata.get("chapter", "")
    if chapter:
        parts.append(chapter)
    
    article = doc.metadata.get("article", "")
    if article:
        parts.append(article)
    
    page = doc.metadata.get("page_number", "")
    if page:
        parts.append(f"Hlm. {page}")
    
    source = doc.metadata.get("source_file", "")
    if source:
        parts.append(f"({source})")
    
    return " | ".join(parts) if parts else "Sumber tidak diketahui"

print("Fungsi format_citation() siap.")

## 14. Full RAG Pipeline

Menggabungkan semua komponen dalam satu pipeline: HyDE → Ensemble Retrieve → Parent-Child → Reranker → DuckDuckGo Fallback → Generate → Citation.

In [ ]:
def rag_pipeline(
    query: str,
    use_hyde: bool = True,
    metadata_filter: Optional[Dict[str, str]] = None,
    verbose: bool = True,
) -> Dict[str, any]:
    """
    Full RAG Pipeline — Advanced Level.
    
    Flow:
    1. HyDE: Generate 2 jawaban halusinasi + gabung embedding
    2. Ensemble Retrieve: BM25 (0.3) + Semantic (0.7), top_k=5
    3. Parent-Child: Ambil parent chunks untuk konteks
    4. Reranker: Urutkan ulang, ambil Top-3
    5. Relevance Check: Jika Top-1 score < 0.5 → DuckDuckGo
    6. Generate: LLM menjawab dengan konteks
    7. Citation: Format sumber
    """
    result = {
        "query": query,
        "answer": "",
        "sources": [],
        "citations": [],
        "from_internet": False,
        "hyde_answers": [],
        "top_reranker_score": None,
        "retrieved_count": 0,
    }
    
    if verbose:
        print(f"[RAG Pipeline] Query: {query[:100]}...")
    
    # Step 1-2: HyDE + Ensemble Retrieve
    if use_hyde:
        if verbose:
            print("[Step 1-2] HyDE + Ensemble Retrieve...")
        child_results, hyde_answers = hyde_retrieve(query, top_k=5)
        result["hyde_answers"] = hyde_answers
    else:
        if verbose:
            print("[Step 2] Ensemble Retrieve...")
        child_results = ensemble_retrieve(query, top_k=5, metadata_filter=metadata_filter)
    
    result["retrieved_count"] = len(child_results)
    
    if verbose:
        print(f"  Retrieved {len(child_results)} child chunks")
    
    # Step 3: Parent-Child
    if verbose:
        print("[Step 3] Parent-Child Retrieve...")
    parent_results = parent_child_retrieve(child_results)
    
    if verbose:
        print(f"  Mapped to {len(parent_results)} parent chunks")
    
    # Step 4: Reranker
    if verbose:
        print("[Step 4] Reranking...")
    reranked_docs, reranker_scores = rerank_documents(query, parent_results, top_k=3)
    
    if reranker_scores:
        result["top_reranker_score"] = reranker_scores[0]
        if verbose:
            print(f"  Top-3 scores: {[f'{s:.3f}' for s in reranker_scores]}")
    
    # Step 5: Relevance Check → DuckDuckGo Fallback
    context = ""
    
    if reranker_scores and reranker_scores[0] < RELEVANCE_THRESHOLD:
        if verbose:
            print(f"[Step 5] Top-1 score ({reranker_scores[0]:.3f}) < threshold ({RELEVANCE_THRESHOLD})")
            print("  → Falling back to DuckDuckGo Search...")
        
        # DuckDuckGo fallback
        web_results = duckduckgo_search(query, max_results=3)
        
        if web_results:
            result["from_internet"] = True
            context_parts = []
            for i, wr in enumerate(web_results):
                context_parts.append(f"[Sumber Web {i+1}: {wr.get('title', 'N/A')}]\n{wr.get('body', '')}")
                result["citations"].append(f"Web: {wr.get('title', 'N/A')} — {wr.get('href', '')}")
            context = "\n\n".join(context_parts)
            if verbose:
                print(f"  Retrieved {len(web_results)} web results")
        else:
            if verbose:
                print("  No web results found. Using empty context.")
            context = "Tidak ada informasi relevan ditemukan di database lokal maupun internet."
    else:
        if verbose:
            print(f"[Step 5] Top-1 score ({reranker_scores[0]:.3f}) >= threshold ({RELEVANCE_THRESHOLD})")
            print("  → Using local documents")
        
        # Gunakan dokumen lokal
        context_parts = []
        for i, doc in enumerate(reranked_docs):
            citation = format_citation(doc)
            context_parts.append(f"[Dokumen {i+1}: {citation}]\n{doc.page_content}")
            result["citations"].append(citation)
            result["sources"].append({
                "citation": citation,
                "reranker_score": doc.metadata.get("reranker_score", 0),
                "content_preview": doc.page_content[:300],
            })
        context = "\n\n".join(context_parts)
    
    # Step 6: Generate Answer
    if verbose:
        print("[Step 6] Generating answer...")
    
    # Bangun prompt dengan context + question
    system_msg = (
        "Anda adalah asisten hukum AI yang membantu tim legal perusahaan. "
        "Jawablah pertanyaan berdasarkan KONTEKS dokumen yang diberikan. "
        "Jika konteks tidak mengandung informasi yang cukup, akui keterbatasan Anda. "
        "Jangan membuat jawaban spekulatif. Selalu gunakan Bahasa Indonesia yang baik dan benar.\n\n"
        "Jika informasi berasal dari internet, sebutkan sumbernya."
    )
    
    user_msg = f"""Konteks:\n{context}\n\nPertanyaan: {query}\n\nJawaban:"""
    
    messages = [
        {"from": "system", "value": system_msg},
        {"from": "user", "value": user_msg},
    ]
    
    inputs = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    
    outputs = llm_model.generate(
        input_ids=inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=llm_tokenizer.pad_token_id,
    )
    
    answer = llm_tokenizer.decode(
        outputs[0][inputs.shape[1]:],
        skip_special_tokens=True,
    )
    
    result["answer"] = answer.strip()
    
    if verbose:
        print(f"[RAG Pipeline] Complete. Answer length: {len(answer)} chars")
    
    return result

print("Full RAG pipeline siap.")

## 15. Test Pipeline — Quick Check

In [ ]:
# Test pipeline dengan beberapa pertanyaan
test_queries = [
    "Apa yang dimaksud dengan PKWT dan PKWTT?",
    "Bagaimana ketentuan upah lembur menurut undang-undang yang berlaku?",
    "Apa hak-hak pekerja perempuan terkait cuti melahirkan?",
]

for query in test_queries[:1]:  # Test 1 query dulu untuk cek
    print("=" * 60)
    print(f"Query: {query}")
    print("=" * 60)
    
    result = rag_pipeline(query, use_hyde=True, verbose=True)
    
    print(f"\nJawaban:\n{result['answer'][:500]}..." if len(result['answer']) > 500 else f"\nJawaban:\n{result['answer']}")
    
    if result["citations"]:
        print(f"\nSumber ({'Internet' if result['from_internet'] else 'Dokumen Lokal'}):")
        for i, cite in enumerate(result["citations"]):
            print(f"  {i+1}. {cite}")
    
    print()

In [ ]:
# TEST CASE WAJIB dari ketentuan Advanced
MANDATORY_QUERY = "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"

print("=" * 70)
print("TEST CASE WAJIB — RAG Pipeline")
print("=" * 70)

result = rag_pipeline(MANDATORY_QUERY, use_hyde=True, verbose=True)

print("\n" + "=" * 70)
print("HASIL AKHIR")
print("=" * 70)

# Cek apakah ada <think> tag (dari model GRPO)
if "<think>" in result["answer"]:
    print("[OK] Model menggunakan format <think> reasoning")
    think_part = result["answer"].split("<think>", 1)[1]
    if "</think>" in think_part:
        think_content = think_part.split("</think>", 1)[0]
        answer_part = think_part.split("</think>", 1)[1] if "</think>" in think_part else think_part
        print(f"[OK] Reasoning length: {len(think_content)} chars")
        print(f"\n[REASONING]:\n{think_content[:300]}..." if len(think_content) > 300 else f"\n[REASONING]:\n{think_content}")
        print(f"\n[JAWABAN]:\n{answer_part[:500]}" if len(answer_part) > 500 else f"\n[JAWABAN]:\n{answer_part}")
    else:
        print(result["answer"])
else:
    print(result["answer"])

print("\n" + "=" * 70)
print("SUMBER:")
print("=" * 70)
for i, cite in enumerate(result["citations"]):
    print(f"  {i+1}. {cite}")
if result["from_internet"]:
    print("  (Informasi dari pencarian internet)")
else:
    print("  (Informasi dari dokumen UU lokal)")

## 16. Gradio Chat Interface

Membungkus pipeline RAG dalam Gradio `ChatInterface`.

In [ ]:
import gradio as gr

def chatbot_response(message: str, history: list) -> str:
    """
    Callback untuk Gradio ChatInterface.
    
    Args:
        message: Pesan user terbaru
        history: Riwayat chat (list of [user_msg, bot_msg] pairs)
    
    Returns:
        str: Response bot
    """
    # Jalankan RAG pipeline (non-verbose untuk UI)
    result = rag_pipeline(message, use_hyde=True, verbose=False)
    
    # Format response dengan citations
    answer = result["answer"]
    
    # Tambahkan sumber di akhir jawaban
    if result["citations"]:
        source_label = "Sumber Internet" if result["from_internet"] else "Sumber Dokumen"
        answer += f"\n\n---\n**[{source_label}]**\n"
        for i, cite in enumerate(result["citations"]):
            answer += f"- {cite}\n"
    
    if result["top_reranker_score"] is not None:
        answer += f"\n> *Relevance Score: {result['top_reranker_score']:.3f}*"
    
    return answer

# Buat Gradio ChatInterface
chat_interface = gr.ChatInterface(
    fn=chatbot_response,
    title="Asisten Hukum Indonesia",
    description="Chatbot AI untuk tanya-jawab seputar hukum ketenagakerjaan dan regulasi Indonesia. Didukung oleh RAG + LLM Fine-Tuned.",
    theme="soft",
    examples=[
        "Apa saja hak pekerja tetap menurut UU yang berlaku?",
        "Bagaimana ketentuan upah minimum?",
        "Apa prosedur PHK yang sah secara hukum?",
        "Saya staf admin, kemarin lembur 3 jam. Apakah saya berhak dapat uang lembur?",
    ],
    cache_examples=False,
)

print("Gradio ChatInterface siap dijalankan.")

In [ ]:
# Jalankan Gradio interface
chat_interface.launch(
    share=False,  # Set True untuk public link (Colab: True)
    debug=False,
)

## 17. Interactive Python Loop (Alternatif)

Jika Gradio tidak tersedia atau untuk testing cepat, gunakan interactive loop dengan `IPython.display.Markdown`.

In [ ]:
# Interactive Python Loop — alternatif jika Gradio tidak bisa digunakan
from IPython.display import Markdown, display

def interactive_loop():
    """
    Interactive loop untuk testing RAG pipeline.
    Ketik 'exit' atau 'quit' untuk keluar.
    """
    print("=" * 60)
    print("Asisten Hukum Indonesia — Interactive Mode")
    print("Ketik 'exit' untuk keluar, 'sources' untuk lihat sumber")
    print("=" * 60)
    
    last_result = None
    
    while True:
        try:
            query = input("\nAnda: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nSampai jumpa!")
            break
        
        if not query:
            continue
        
        if query.lower() in ["exit", "quit", "keluar"]:
            print("Sampai jumpa!")
            break
        
        if query.lower() == "sources" and last_result:
            print("\nSumber terakhir:")
            for i, cite in enumerate(last_result.get("citations", [])):
                print(f"  {i+1}. {cite}")
            continue
        
        # Jalankan pipeline
        result = rag_pipeline(query, use_hyde=True, verbose=False)
        last_result = result
        
        # Tampilkan jawaban dengan Markdown
        answer_md = result["answer"]
        if result["citations"]:
            source_label = "Sumber Internet" if result["from_internet"] else "Sumber Dokumen"
            answer_md += f"\n\n---\n**[{source_label}]**  \n"
            for cite in result["citations"]:
                answer_md += f"- {cite}  \n"
        
        display(Markdown(f"**Asisten:**\n\n{answer_md}"))

# Uncomment untuk menjalankan interactive loop:
# interactive_loop()

print("Interactive loop siap. Uncomment baris terakhir untuk menjalankan.")

## Kesimpulan

Notebook ini menyelesaikan seluruh kriteria RAG:

**Basic (2 pts)** ✅
- [x] PDF loading + text splitting (chunk=512, overlap=128)
- [x] Embedding open-source (BAAI/bge-m3) + ChromaDB
- [x] Model hasil fine-tuning untuk inference
- [x] Interface: Gradio ChatInterface + Interactive Python Loop

**Skilled (3 pts)** ✅
- [x] Metadata enrichment (nomor UU, tahun, bab, pasal)
- [x] Metadata filtering + sitasi
- [x] Ensemble Retriever: BM25 (0.3) + Semantic (0.7), top_k=5
- [x] Parent-Child Retriever (child=512, parent=1024)

**Advanced (4 pts)** ✅
- [x] HyDE: 2 jawaban halusinasi
- [x] Reranker: BAAI/bge-reranker-v2-m3, Top-3
- [x] Relevance score check + DuckDuckGo fallback (threshold=0.5)